In [3]:
import os
import math
import requests
from datetime import datetime
from typing import Optional, Dict

import numpy as np
import pandas as pd
import joblib

# ============================================================
# CONFIG
# ============================================================

PROJ_ROOT = os.getcwd()
MODEL_PATH = os.path.join(
    PROJ_ROOT,
    "artifacts_yield",
    "yield_pipeline_forward_chain_STACKED_PRODUCTION.pkl"
)

PROC_DIR = os.path.join(PROJ_ROOT, "processed_training_csvs")

FOLD_MODELS = None
META_MODEL = None
ENRICHED_DF = None


# ============================================================
# TEXT NORMALIZATION
# ============================================================

def normtext(s: str) -> str:
    return str(s).strip().lower().replace("&", "and").replace(".", "").replace("-", " ")


# ============================================================
# LOAD HISTORICAL DATA
# ============================================================

def load_enriched_base():
    global ENRICHED_DF

    if ENRICHED_DF is not None:
        return ENRICHED_DF

    files = [f for f in os.listdir(PROC_DIR)
             if f.startswith("engineered_training_")]

    dfs = []
    for f in files:
        path = os.path.join(PROC_DIR, f)
        print(f"[INFO] Loading historical data from: {path}")
        dfs.append(pd.read_csv(path))

    df = pd.concat(dfs, ignore_index=True)

    df["statenorm"] = df["statename"].map(normtext)
    df["districtnorm"] = df["districtname"].map(normtext)

    ENRICHED_DF = df

    print(f"[INFO] Combined shape: {df.shape}")
    print(f"[INFO] Years covered: {df['cropyear'].min()} - {df['cropyear'].max()}")

    return df


# ============================================================
# LAT/LON RESOLUTION
# ============================================================

def resolve_lat_lon(state: str, district: str):
    df = load_enriched_base()

    sn = normtext(state)
    dn = normtext(district)

    rows = df[
        (df["statenorm"] == sn) &
        (df["districtnorm"] == dn)
    ]

    if not rows.empty:
        lat = float(rows.iloc[0]["lat"])
        lon = float(rows.iloc[0]["lon"])
        print(f"[DEBUG] Resolved lat/lon to ({lat:.4f}, {lon:.4f})")
        return lat, lon

    print("[WARN] Using default India centroid.")
    return 22.9734, 78.6569


# ============================================================
# NASA WEATHER (CUMULATIVE)
# ============================================================

def fetch_cumulative_weather(lat: float,
                             lon: float,
                             sowing_date: datetime,
                             forecast_date: datetime) -> Dict[str, float]:

    start_str = sowing_date.strftime("%Y%m%d")
    end_str = forecast_date.strftime("%Y%m%d")

    url = "https://power.larc.nasa.gov/api/temporal/daily/point"

    params = {
        "latitude": f"{lat:.4f}",
        "longitude": f"{lon:.4f}",
        "start": start_str,
        "end": end_str,
        "parameters": "PRECTOTCORR,T2M,T2M_MAX,T2M_MIN",
        "community": "ag",
        "format": "JSON"
    }

    try:
        r = requests.get(url, params=params, timeout=30)
        r.raise_for_status()
        data = r.json()

        param = data["properties"]["parameter"]

        rain = param.get("PRECTOTCORR", {})
        tavg = param.get("T2M", {})
        tmax = param.get("T2M_MAX", {})
        tmin = param.get("T2M_MIN", {})

        df = pd.DataFrame({
            "date": list(rain.keys()),
            "rain": list(rain.values()),
            "tavg": list(tavg.values()),
            "tmax": list(tmax.values()),
            "tmin": list(tmin.values())
        })

        df["date"] = pd.to_datetime(df["date"], format="%Y%m%d")

        for col in ["rain", "tavg", "tmax", "tmin"]:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            df.loc[df[col] < -900, col] = np.nan

        df = df.dropna()

        if df.empty:
            print("[WARN] NASA returned empty dataset.")
            return {}

        rainfall_sum = float(df["rain"].sum())
        tavg_mean = float(df["tavg"].mean())
        tmax_mean = float(df["tmax"].mean())
        tmin_mean = float(df["tmin"].mean())

        et0_sum = 0.0
        for _, row in df.iterrows():
            et0_sum += 0.0023 * max(0, row["tavg"] - 17.8) * \
                       math.sqrt(max(0, row["tmax"] - row["tmin"]))

        gdd_sum = float(
            ((df["tmax"] + df["tmin"]) / 2 - 10)
            .clip(lower=0)
            .sum()
        )

        print(f"[INFO] NASA Weather Loaded: {len(df)} days")

        return {
            "Rainfall_sum": rainfall_sum,
            "Tavg_mean": tavg_mean,
            "Tmax_mean": tmax_mean,
            "Tmin_mean": tmin_mean,
            "ET0_sum": et0_sum,
            "GDD_sum": gdd_sum
        }

    except Exception as e:
        print(f"[ERROR] NASA fetch failed: {e}")
        return {}


# ============================================================
# SAME-SEASON BASELINE (Rolling 3 Years)
# ============================================================

def get_historical_lags(state: str,
                        district: str,
                        crop: str,
                        season: str,
                        year: int,
                        k: int = 3):

    df = load_enriched_base()

    sn = normtext(state)
    dn = normtext(district)
    crop_norm = crop.lower().strip()

    hist = df[
        (df["statenorm"] == sn) &
        (df["districtnorm"] == dn) &
        (df["crop"].str.lower().str.strip() == crop_norm) &
        (df["season"] == season) &
        (df["cropyear"] < year)
    ].sort_values("cropyear", ascending=False)

    if hist.empty:
        print("[WARN] No same-season baseline found.")
        return {}

    hist = hist.head(k)

    print(f"[INFO] Baseline from years: {hist['cropyear'].tolist()}")

    baseline = {}
    base_cols = [
        "yieldcalc",
        "production",
        "area",
        "Rainfall_sum",
        "Tavg_mean",
        "Tmax_mean",
        "Tmin_mean",
        "ET0_sum",
        "GDD_sum"
    ]

    for col in base_cols:
        baseline[f"{col}_lag1"] = float(hist[col].mean()) \
            if col in hist.columns else 0.0

    return baseline


# ============================================================
# SEASON INFO
# ============================================================

def calculate_season_info(crop, sowing_date, forecast_date):
    days_total = 120
    days_elapsed = (forecast_date - sowing_date).days
    progress = max(0.0, min(1.0, days_elapsed / days_total))

    month = sowing_date.month
    if 6 <= month <= 10:
        season = "Kharif"
    elif month in [11, 12, 1, 2]:
        season = "Rabi"
    else:
        season = "Zaid"

    return {
        "season_inferred": season,
        "progress": progress
    }


# ============================================================
# CONFIDENCE SCALING
# ============================================================

def calculate_confidence_scaling(season_progress: float) -> float:
    if season_progress < 0.3:
        return 0.60 + season_progress * 0.80
    if season_progress < 0.6:
        return 0.84 + season_progress * 0.20
    return min(1.0, 0.96 + season_progress * 0.04)


# ============================================================
# LOAD MODEL
# ============================================================

def load_model():
    global FOLD_MODELS, META_MODEL

    artifacts = joblib.load(MODEL_PATH)
    models_forward_chain = artifacts["models_forward_chain"]

    FOLD_MODELS = models_forward_chain["yieldcalc"]["fold_models"]
    META_MODEL = models_forward_chain["yieldcalc"]["meta_model"]

    print(f"[MODEL] Loaded {len(FOLD_MODELS)} fold models.")


# ============================================================
# MAIN PREDICTION FUNCTION
# ============================================================

def predict_yield(state: str,
                  district: str,
                  crop: str,
                  area: float,
                  sowing_date_str: str,
                  forecast_date_str: Optional[str] = None):

    sowing_date = datetime.strptime(sowing_date_str, "%Y-%m-%d")
    forecast_date = datetime.strptime(
        forecast_date_str, "%Y-%m-%d"
    ) if forecast_date_str else datetime.now()

    year = sowing_date.year

    season_info = calculate_season_info(
        crop, sowing_date, forecast_date
    )

    lat, lon = resolve_lat_lon(state, district)

    current_weather = fetch_cumulative_weather(
        lat, lon, sowing_date, forecast_date
    )

    lags = get_historical_lags(
        state, district, crop,
        season_info["season_inferred"],
        year
    )

    features = {
        "statename": state,
        "districtname": district,
        "crop": crop,
        "area": area,
        "cropyear": year,
        "season": season_info["season_inferred"],
        "lat": lat,
        "lon": lon,
        "statenorm": normtext(state),
        "districtnorm": normtext(district)
    }

    # Intelligent weather fallback
    weather_cols = [
        "Rainfall_sum",
        "Tavg_mean",
        "Tmax_mean",
        "Tmin_mean",
        "ET0_sum",
        "GDD_sum"
    ]

    for col in weather_cols:
        nasa_val = current_weather.get(col)
        if nasa_val is None:
            fallback = lags.get(f"{col}_lag1", 0.0)
            features[col] = fallback
            print(f"[INFO] NASA missing {col} → using baseline")
        else:
            features[col] = nasa_val

    base_cols = [
        "yieldcalc",
        "production",
        "area",
        "Rainfall_sum",
        "Tavg_mean",
        "Tmax_mean",
        "Tmin_mean",
        "ET0_sum",
        "GDD_sum"
    ]

    for col in base_cols:
        lag_col = f"{col}_lag1"
        delta_col = f"{col}_delta1"

        lag_val = lags.get(lag_col, 0.0)
        current_val = features.get(col, 0.0)

        features[lag_col] = lag_val
        features[delta_col] = current_val - lag_val

    print("\n--- [DEBUG] FINAL FEATURES ---")
    for k, v in sorted(features.items()):
        print(f"{k:<25}: {v}")
    print("--------------------------------------\n")

    input_data = pd.DataFrame([features])

    base_preds = np.array(
        [model.predict(input_data)[0] for model in FOLD_MODELS]
    )

    fold_mean = np.mean(base_preds)
    fold_std = np.std(base_preds)

    meta_features = np.hstack([
        fold_mean.reshape(1, -1),
        base_preds.reshape(1, -1)
    ])

    yield_pred_raw = float(
        META_MODEL.predict(meta_features)[0]
    )

    confidence_scaling = calculate_confidence_scaling(
        season_info["progress"]
    )

    yield_pred_scaled = yield_pred_raw * confidence_scaling

    final_yield = max(0.5, min(yield_pred_scaled, 15.0))

    uncertainty = max(fold_std * 2,
                      final_yield * 0.20)

    return {
        "yield_per_hectare": final_yield,
        "yield_range": (
            max(0.1, final_yield - uncertainty),
            final_yield + uncertainty
        ),
        "total_production": final_yield * area,
        "season_info": season_info,
        "yield_pred_raw": yield_pred_raw,
        "confidence_scaling": confidence_scaling,
        "model_confidence":
            1 - (fold_std / fold_mean)
            if fold_mean else 0.0,
        "uncertainty": uncertainty
    }


# ============================================================
# EXAMPLE USAGE
# ============================================================

if __name__ == "__main__":

    load_model()

    print("\n" + "="*80)
    print("RUNNING EXAMPLE PREDICTION")
    print("="*80)

    prediction_results = predict_yield(
        state="Punjab",
        district="Ludhiana",
        crop="Bajra",
        area=5.0,
        sowing_date_str="2026-01-27",
        forecast_date_str="2026-03-02"
    )

    print("\n" + "="*80)
    print("PREDICTION RESULTS")
    print("="*80)

    print(f"Raw Model Prediction:          {prediction_results['yield_pred_raw']:.2f} t/ha")
    print(f"Season Progress:               {prediction_results['season_info']['progress']:.1%}")
    print(f"Confidence Scaling Factor:     {prediction_results['confidence_scaling']:.1%}")
    print("-" * 40)
    print(f"Final Scaled Yield:            {prediction_results['yield_per_hectare']:.2f} t/ha")
    print("-" * 40)
    print(f"Model Confidence (Agreement):  {prediction_results['model_confidence']:.1%}")
    print(f"Uncertainty Value:             +/- {prediction_results['uncertainty']:.2f} t/ha")
    print(f"Predicted Range:               {prediction_results['yield_range'][0]:.2f} - {prediction_results['yield_range'][1]:.2f} t/ha")
    print("="*80)

[MODEL] Loaded 20 fold models.

RUNNING EXAMPLE PREDICTION
[INFO] Loading historical data from: C:\Users\Ritvik Bhat\agri ai web app\processed_training_csvs\engineered_training_2010.csv
[INFO] Loading historical data from: C:\Users\Ritvik Bhat\agri ai web app\processed_training_csvs\engineered_training_2015.csv
[INFO] Loading historical data from: C:\Users\Ritvik Bhat\agri ai web app\processed_training_csvs\engineered_training_2023.csv
[INFO] Combined shape: (2868922, 48)
[INFO] Years covered: 1997 - 2023
[DEBUG] Resolved lat/lon to (30.9090, 75.8516)
[INFO] NASA Weather Loaded: 29 days
[WARN] No same-season baseline found.

--- [DEBUG] FINAL FEATURES ---
ET0_sum                  : 0.3643154928342768
ET0_sum_delta1           : 0.3643154928342768
ET0_sum_lag1             : 0.0
GDD_sum                  : 247.03500000000003
GDD_sum_delta1           : 247.03500000000003
GDD_sum_lag1             : 0.0
Rainfall_sum             : 16.87
Rainfall_sum_delta1      : 16.87
Rainfall_sum_lag1       

In [ ]:
import os
import pandas as pd
import json

print("=" * 80)
print("FILE PATH DIAGNOSTICS - What Your Notebook Loads")
print("=" * 80)

# 1. Check Model Path
MODEL_PATH = r"C:\Users\Ritvik Bhat\agri ai web app\artifacts_yield\yield_pipeline_forward_chain_STACKED_PRODUCTION.pkl"
print(f"\n1. MODEL FILE:")
print(f"   Path: {MODEL_PATH}")
print(f"   Exists: {os.path.isfile(MODEL_PATH)}")

# 2. Check Crop Durations Path
CROP_DURATIONS_PATH = r"C:\Users\Ritvik Bhat\agri ai web app\crop_durations.json"
print(f"\n2. CROP DURATIONS FILE:")
print(f"   Path: {CROP_DURATIONS_PATH}")
print(f"   Exists: {os.path.isfile(CROP_DURATIONS_PATH)}")

# 3. Check Historical Data Directory
PROC_DIR = r"C:\Users\Ritvik Bhat\training_csvs"
print(f"\n3. HISTORICAL DATA DIRECTORY:")
print(f"   Path: {PROC_DIR}")
print(f"   Exists: {os.path.isdir(PROC_DIR)}")

if os.path.isdir(PROC_DIR):
    # List all run_ folders
    subdirs = [d for d in os.listdir(PROC_DIR) if d.startswith("run_")]
    print(f"   Found {len(subdirs)} 'run_' folders:")
    for d in sorted(subdirs):
        print(f"     - {d}")
    
    # Check for the enriched base file
    if subdirs:
        subdirs_full = [os.path.join(PROC_DIR, d) for d in subdirs]
        subdirs_full.sort()
        latest_run = subdirs_full[-1]
        enriched_path = os.path.join(latest_run, "01_enriched_base.csv")
        
        print(f"\n   Latest run folder: {os.path.basename(latest_run)}")
        print(f"   Enriched data file: 01_enriched_base.csv")
        print(f"   Full path: {enriched_path}")
        print(f"   Exists: {os.path.isfile(enriched_path)}")
        
        if os.path.isfile(enriched_path):
            # Load and show info about the CSV
            df = pd.read_csv(enriched_path)
            print(f"\n   CSV Info:")
            print(f"     - Shape: {df.shape}")
            print(f"     - Columns: {list(df.columns[:15])}...")
            print(f"     - Year range: {df['cropyear'].min() if 'cropyear' in df.columns else 'N/A'} to {df['cropyear'].max() if 'cropyear' in df.columns else 'N/A'}")
            
            # Check for Ludhiana, Punjab, Wheat, 2024 data
            if all(col in df.columns for col in ['statename', 'districtname', 'crop', 'cropyear']):
                df['statenorm'] = df['statename'].str.strip().str.lower()
                df['districtnorm'] = df['districtname'].str.strip().str.lower()
                df['cropnorm'] = df['crop'].str.strip().str.lower()
                
                test_data = df[
                    (df['statenorm'] == 'punjab') & 
                    (df['districtnorm'] == 'ludhiana') & 
                    (df['cropnorm'] == 'wheat') & 
                    (df['cropyear'] == 2024)
                ]
                
                print(f"\n   Test Query (Ludhiana, Punjab, Wheat, 2024):")
                print(f"     - Found: {len(test_data)} rows")
                if len(test_data) > 0:
                    print(f"     - Sample columns: lat={test_data.iloc[0].get('lat', 'N/A')}, lon={test_data.iloc[0].get('lon', 'N/A')}")
                    print(f"     - Weather data: Rainfall_sum={test_data.iloc[0].get('Rainfall_sum', 'N/A')}")

print("\n" + "=" * 80)
print("SUMMARY:")
print("=" * 80)
if os.path.isdir(PROC_DIR) and subdirs:
    print("Your notebook loads historical data from:")
    print(f"  {enriched_path}")
    print("\nYour app.py should use:")
    print(f"  ENRICHED_DATA_PATH = r\"{enriched_path}\"")
else:
    print("Historical data directory not found or no run folders exist.")
print("=" * 80)


FILE PATH DIAGNOSTICS - What Your Notebook Loads

1. MODEL FILE:
   Path: C:\Users\Ritvik Bhat\artifacts_yield\yield_pipeline_forward_chain_STACKED_PRODUCTION.pkl
   Exists: True

2. CROP DURATIONS FILE:
   Path: C:\Users\Ritvik Bhat\crop_durations.json
   Exists: False

3. HISTORICAL DATA DIRECTORY:
   Path: C:\Users\Ritvik Bhat\training_csvs
   Exists: False

SUMMARY:
Historical data directory not found or no run folders exist.
